In [1]:
import numpy as np
import pandas as pd
import os

rng = np.random.default_rng(42)

N = 45  # small-scale course survey, consistent with a 5-page report / 15-week course

# --- Demographics -----------------------------------------------------

age_group = rng.choice(
    ["18-24", "25-34", "35-44", "45+"],
    size=N,
    p=[0.30, 0.35, 0.20, 0.15]
)

gender = rng.choice(
    ["Male", "Female", "Prefer not to say"],
    size=N,
    p=[0.5, 0.45, 0.05]
)

education = rng.choice(
    ["Bachelor's", "Master's", "Doctorate", "Other"],
    size=N,
    p=[0.55, 0.35, 0.05, 0.05]
)

technical_background = rng.choice(
    ["Yes", "No"],
    size=N,
    p=[0.47, 0.53]
)

# --- AI exposure --------------------------------------------------------

ai_familiarity = rng.integers(1, 6, size=N)
# 1 = not at all familiar, 5 = very familiar

usage_frequency = rng.choice(
    ["Daily", "Weekly", "Rarely", "Never"],
    size=N,
    p=[0.30, 0.35, 0.25, 0.10]
)

usage_map = {
    "Daily": 4,
    "Weekly": 3,
    "Rarely": 2,
    "Never": 1
}

usage_score = np.array([
    usage_map[u] for u in usage_frequency
])

age_map = {
    "18-24": 1,
    "25-34": 2,
    "35-44": 3,
    "45+": 4
}

age_score = np.array([
    age_map[a] for a in age_group
])

tech_score = np.array([
    1 if t == "Yes" else 0
    for t in technical_background
])

# --- Likert attitude items ---------------------------------------------
# 1 = Strongly Disagree, 5 = Strongly Agree

def bounded_likert(base):
    noise = rng.normal(0, 0.8, size=N)
    val = np.clip(np.round(base + noise), 1, 5)
    return val.astype(int)


q1_low_stakes_trust = bounded_likert(
    2.6 + 0.35 * ai_familiarity - 0.15 * age_score
)

q2_high_stakes_trust = bounded_likert(
    1.6 + 0.30 * ai_familiarity
    - 0.30 * age_score
    + 0.2 * tech_score
)

q3_transparency = bounded_likert(
    2.2 + 0.28 * ai_familiarity - 0.10 * age_score
)

q4_would_follow_ai = bounded_likert(
    1.8 + 0.30 * ai_familiarity
    + 0.25 * tech_score
    - 0.15 * age_score
)

q5_bias_concern = bounded_likert(
    3.4 - 0.15 * ai_familiarity
    + 0.20 * age_score
)
# Reverse-coded concern item

q6_ai_more_objective = bounded_likert(
    2.3 + 0.32 * ai_familiarity
    - 0.10 * age_score
    + 0.15 * tech_score
)

# --- Composite Trust Score & binary label -------------------------------

# Concern item reverse-scored before combining (6 - value)

trust_composite = (
    q1_low_stakes_trust
    + q2_high_stakes_trust
    + q3_transparency
    + q4_would_follow_ai
    + q6_ai_more_objective
    + (6 - q5_bias_concern)
) / 6.0

trust_threshold = float(
    np.median(trust_composite)
)

trust_label = np.where(
    trust_composite >= trust_threshold,
    "High Trust",
    "Low Trust"
)

# --- Create dataset -----------------------------------------------------

df = pd.DataFrame({
    "respondent_id": [
        f"R{str(i + 1).zfill(3)}"
        for i in range(N)
    ],

    "age_group": age_group,

    "gender": gender,

    "education_level": education,

    "technical_background": technical_background,

    "ai_familiarity_1to5": ai_familiarity,

    "usage_frequency": usage_frequency,

    "q1_trust_low_stakes": q1_low_stakes_trust,

    "q2_trust_high_stakes": q2_high_stakes_trust,

    "q3_perceived_transparency": q3_transparency,

    "q4_would_follow_ai_over_own_judgment": q4_would_follow_ai,

    "q5_bias_concern": q5_bias_concern,

    "q6_ai_more_objective_than_human": q6_ai_more_objective,

    "trust_composite_score": trust_composite.round(2),

    "trust_label": trust_label,
})

# --- Save dataset -------------------------------------------------------

os.makedirs("data", exist_ok=True)

df.to_csv(
    "data/ai_trust_survey_synthetic.csv",
    index=False
)

# --- Display results ----------------------------------------------------

print(df.head(10).to_string())

print("\nShape:", df.shape)

print(
    "\nLabel balance:\n",
    df["trust_label"].value_counts()
)

  respondent_id age_group  gender education_level technical_background  ai_familiarity_1to5 usage_frequency  q1_trust_low_stakes  q2_trust_high_stakes  q3_perceived_transparency  q4_would_follow_ai_over_own_judgment  q5_bias_concern  q6_ai_more_objective_than_human  trust_composite_score trust_label
0          R001     35-44  Female      Bachelor's                  Yes                    5           Daily                    3                     2                          5                                     5                3                                3                   3.50  High Trust
1          R002     25-34    Male        Master's                   No                    5           Daily                    4                     3                          4                                     3                2                                3                   3.50  High Trust
2          R003       45+    Male      Bachelor's                   No                    2      